# 🍜 Vietnamese Food AI — Nhận diện món ăn Việt Nam bằng AI

**Bài toán:** Image Classification (phân loại ảnh) — cho 1 tấm ảnh món ăn Việt Nam, AI đoán xem đó là món gì trong 30 món.

**Công nghệ dùng:**
- Python + TensorFlow/Keras
- **Transfer Learning** với mạng **MobileNetV2** (đã học sẵn cách "nhìn" ảnh từ 1 triệu ảnh ImageNet, mình chỉ cần "dạy thêm" nó nhận món ăn Việt Nam)
- Chạy trên Google Colab (có GPU miễn phí)
- Sau khi train xong → lưu model → dùng trong web Streamlit (file `app.py` đi kèm)

**Dataset:** [Vietnamese Foods (Kaggle)](https://www.kaggle.com/datasets/quandang/vietnamese-foods) — 30 loại món ăn, đã chia sẵn Train/Validate/Test.

---

### 📌 Mình đã sửa gì so với bản gốc của em?

Mình đọc code em gửi, đây là 2 vấn đề chính đã sửa, em nên hiểu rõ 2 điều này vì nó rất hay gặp khi đi làm thật:

1. **Overfitting nặng** — model gốc của em dùng CNN tự xây từ đầu (Conv2D → Conv2D → Dense), không có Data Augmentation, không có Dropout. Kết quả: train accuracy lên tới **99.8%** nhưng val accuracy chỉ dừng ở **~15%**. Đây là dấu hiệu kinh điển của **overfitting** — model "học thuộc lòng" ảnh train thay vì học đặc điểm chung của món ăn. → Notebook mới dùng **Transfer Learning (MobileNetV2)** + **Data Augmentation** + **Dropout** + **EarlyStopping** để giải quyết.
2. **Lỗi `You must call compile() before using the model`** — lỗi này xảy ra vì em chạy lại cell tạo `model = Sequential([...])` (định nghĩa model mới) SAU KHI đã compile, nhưng chưa chạy lại cell `model.compile(...)`. Trong Jupyter/Colab, thứ tự **chạy cell** mới là thứ quyết định, không phải thứ tự cell nằm trên/dưới trong notebook. Mẹo: mỗi khi sửa lại kiến trúc model, luôn dùng **Runtime → Run all** hoặc chạy lại đúng theo thứ tự từ trên xuống để tránh lệch trạng thái.

---


## Mục lục
1. Cài đặt & Import thư viện
2. Tải dataset từ Kaggle
3. Khám phá dữ liệu (EDA)
4. Chuẩn bị Data Pipeline (Augmentation)
5. Xây dựng model — Transfer Learning MobileNetV2
6. Compile & Callbacks
7. Huấn luyện — Giai đoạn 1 (Feature Extraction)
8. Huấn luyện — Giai đoạn 2 (Fine-tuning)
9. Trực quan hoá quá trình huấn luyện
10. Đánh giá model trên tập Test
11. Lưu model (để dùng cho Streamlit)
12. Demo dự đoán 1 ảnh
13. Tổng kết & bước tiếp theo


## 1. Cài đặt & Import thư viện

Giải thích nhanh từng thư viện dùng làm gì:

| Thư viện | Dùng để làm gì |
|---|---|
| `kagglehub` | Tải dataset trực tiếp từ Kaggle về Colab |
| `tensorflow` / `keras` | Xây dựng và train mô hình AI |
| `matplotlib`, `seaborn` | Vẽ biểu đồ (accuracy/loss, confusion matrix...) |
| `numpy`, `pandas` | Xử lý số liệu, bảng dữ liệu |
| `sklearn` | Tính các chỉ số đánh giá (precision, recall, f1-score, confusion matrix) |


In [ ]:
!pip install -q kagglehub

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow version:", tf.__version__)
print("GPU khả dụng:", tf.config.list_physical_devices('GPU'))


> ⚠️ **Lưu ý quan trọng:** nhớ bật GPU trước khi train, không thì mỗi epoch sẽ mất rất lâu (như bản gốc của em, ~30 phút/epoch trên CPU).
> Vào **Runtime → Change runtime type → Hardware accelerator → GPU (T4)** rồi mới chạy tiếp.

## 2. Tải dataset từ Kaggle

`kagglehub.dataset_download()` sẽ tự động tải và giải nén dataset về máy ảo Colab. Dataset gồm 3 thư mục con: `Train`, `Validate`, `Test`, mỗi thư mục lại có 30 thư mục con tương ứng 30 loại món ăn (mỗi thư mục chứa ảnh của đúng món đó — đây gọi là cấu trúc **"ImageFolder"**, rất phổ biến cho bài toán classification).

In [ ]:
import kagglehub

# Tải phiên bản mới nhất của dataset
path = kagglehub.dataset_download("quandang/vietnamese-foods")
dataset_path = path

print("Dataset đã tải về:", dataset_path)
print("Nội dung:", os.listdir(dataset_path))


In [ ]:
train_dir = os.path.join(dataset_path, "Images", "Train")
val_dir   = os.path.join(dataset_path, "Images", "Validate")
test_dir  = os.path.join(dataset_path, "Images", "Test")

class_names = sorted(os.listdir(train_dir))
NUM_CLASSES = len(class_names)

print("Số lớp (số loại món ăn):", NUM_CLASSES)
print(class_names)


## 3. Khám phá dữ liệu (EDA — Exploratory Data Analysis)

Trước khi train, luôn nên kiểm tra 2 điều:
1. **Số lượng ảnh mỗi lớp có đều nhau không?** (nếu lệch nhiều → mất cân bằng dữ liệu / "class imbalance", model dễ thiên vị lớp nhiều ảnh hơn)
2. **Ảnh trông như thế nào?** (kiểm tra bằng mắt xem ảnh có lỗi, bị mờ, sai nhãn không)

In [ ]:
def count_images_per_class(base_dir):
    counts = {}
    for cls in sorted(os.listdir(base_dir)):
        cls_path = os.path.join(base_dir, cls)
        counts[cls] = len(os.listdir(cls_path))
    return counts

train_counts = count_images_per_class(train_dir)
val_counts   = count_images_per_class(val_dir)
test_counts  = count_images_per_class(test_dir)

df_counts = pd.DataFrame({
    "Train": train_counts,
    "Validate": val_counts,
    "Test": test_counts
})
df_counts["Total"] = df_counts.sum(axis=1)
df_counts = df_counts.sort_values("Total", ascending=False)

print("Tổng số ảnh: Train =", sum(train_counts.values()),
      "| Validate =", sum(val_counts.values()),
      "| Test =", sum(test_counts.values()))
df_counts


In [ ]:
# Vẽ biểu đồ số ảnh/lớp trong tập Train để xem có bị mất cân bằng không
plt.figure(figsize=(14, 6))
df_counts["Train"].sort_values().plot(kind="barh", color="#e07a5f")
plt.title("Số lượng ảnh Train theo từng món ăn")
plt.xlabel("Số ảnh")
plt.tight_layout()
plt.show()

print("Món có ít ảnh Train nhất:", df_counts['Train'].idxmin(), "-", df_counts['Train'].min(), "ảnh")
print("Món có nhiều ảnh Train nhất:", df_counts['Train'].idxmax(), "-", df_counts['Train'].max(), "ảnh")


In [ ]:
# Xem thử vài ảnh mẫu ngẫu nhiên để "mắt thường" kiểm tra chất lượng data
import random
from tensorflow.keras.preprocessing.image import load_img

sample_classes = random.sample(class_names, 6)

plt.figure(figsize=(15, 8))
for i, cls in enumerate(sample_classes):
    cls_dir = os.path.join(train_dir, cls)
    img_name = random.choice(os.listdir(cls_dir))
    img = load_img(os.path.join(cls_dir, img_name), target_size=(224, 224))

    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.title(cls)
    plt.axis("off")

plt.tight_layout()
plt.show()


**Nhận xét:** nếu biểu đồ ở trên cho thấy các lớp không quá lệch (ví dụ lớp nhiều nhất không gấp quá 2-3 lần lớp ít nhất) thì mình không bắt buộc phải xử lý mất cân bằng. Nhưng để an toàn, ở bước train mình vẫn sẽ tính sẵn `class_weight` (trọng số cho từng lớp) — nếu lớp nào ít ảnh hơn, model sẽ "bị phạt nặng hơn" khi đoán sai lớp đó, giúp công bằng giữa các lớp.

## 4. Chuẩn bị Data Pipeline (Augmentation)

### Data Augmentation là gì?
Là kỹ thuật **biến đổi ngẫu nhiên ảnh gốc** (xoay nhẹ, lật ngang, zoom, dịch chuyển...) mỗi lần model nhìn thấy ảnh đó trong lúc train. Mục đích: giúp model không "học thuộc" từng pixel của ảnh train, mà học được **đặc điểm chung** của món ăn (hình dáng, màu sắc, kết cấu...) → giảm overfitting, tăng khả năng nhận diện ảnh mới trong thực tế.

Đây chính là phần **quan trọng nhất bị thiếu** trong bản gốc của em — bản gốc chỉ có `rescale=1./255` (chuẩn hoá giá trị pixel), không hề có augmentation.

> 📝 Lưu ý: augmentation **chỉ áp dụng cho tập Train**. Tập Validate/Test phải giữ nguyên ảnh gốc, vì mục đích của 2 tập này là đánh giá model một cách trung thực, không phải để train.

### Vì sao dùng `preprocess_input` thay vì `rescale=1./255`?
`MobileNetV2` được huấn luyện gốc (trên ImageNet) với ảnh đầu vào được chuẩn hoá theo công thức riêng của nó (đưa giá trị pixel về khoảng **[-1, 1]**), khác với cách chuẩn hoá đơn giản về **[0, 1]** mà bản gốc dùng. Nếu dùng sai kiểu chuẩn hoá, phần "kiến thức" MobileNetV2 đã học từ trước sẽ bị lệch, model học chậm và kém chính xác hơn hẳn. Do đó ta dùng đúng hàm `preprocess_input` đi kèm MobileNetV2.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Tập Train: có augmentation (xoay, lật, zoom, dịch chuyển...)
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,          # xoay ảnh ngẫu nhiên tối đa 20 độ
    width_shift_range=0.15,     # dịch ảnh ngang tối đa 15%
    height_shift_range=0.15,    # dịch ảnh dọc tối đa 15%
    shear_range=0.1,            # bóp méo nhẹ
    zoom_range=0.15,            # zoom in/out tối đa 15%
    horizontal_flip=True,       # lật ngang ảnh (món ăn nhìn ngang vẫn hợp lý)
    fill_mode="nearest"         # cách "vá" phần ảnh trống sau khi xoay/dịch
)

# Tập Validate và Test: KHÔNG augmentation, chỉ chuẩn hoá đúng kiểu MobileNetV2
val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)


In [ ]:
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=42
)

val_data = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_data = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

# class_indices: dict {"ten_mon_an": index}. Ta sẽ cần cái này để lưu lại cho Streamlit dùng sau này
print(train_data.class_indices)


In [ ]:
# Tính class_weight — để model không thiên vị lớp có nhiều ảnh hơn
y_train_labels = train_data.classes  # nhãn (dạng số) của toàn bộ ảnh train

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_labels),
    y=y_train_labels
)
class_weight_dict = dict(enumerate(class_weights_array))
print("Class weights (một vài lớp đầu):", dict(list(class_weight_dict.items())[:5]))


## 5. Xây dựng model — Transfer Learning với MobileNetV2

### Transfer Learning là gì? (giải thích cho người mới)
Thay vì tự dạy AI "nhìn" ảnh từ con số 0 (như bản CNN gốc — rất tốn dữ liệu và dễ overfit khi dataset nhỏ), ta **mượn lại** một mạng đã được Google huấn luyện sẵn trên **1.4 triệu ảnh** thuộc **1000 loại vật thể khác nhau** (dataset ImageNet) — gọi là **MobileNetV2**. Mạng này đã học rất giỏi cách nhận biết các đặc trưng cơ bản của ảnh: cạnh, màu sắc, hoạ tiết, hình khối...

Ta chỉ cần "dạy thêm" cho nó phần cuối: cách phân biệt 30 món ăn Việt Nam cụ thể. Cách này:
- Cần **ít dữ liệu hơn** để đạt độ chính xác tốt
- Train **nhanh hơn**
- **Ít bị overfitting hơn** so với train từ đầu

### Kiến trúc model
```
Ảnh đầu vào (224x224x3)
     ↓
MobileNetV2 (đã học sẵn, ĐÓNG BĂNG - không train lại ở giai đoạn 1)
     ↓
GlobalAveragePooling2D   → nén đặc trưng ảnh thành 1 vector
     ↓
Dropout(0.3)             → tắt ngẫu nhiên 30% neuron mỗi lần train, chống overfitting
     ↓
Dense(128, relu)         → lớp học đặc trưng riêng cho món ăn Việt Nam
     ↓
Dropout(0.2)
     ↓
Dense(30, softmax)       → lớp output: xác suất cho từng món trong 30 món
```

**"Đóng băng" (freeze)** nghĩa là gì? Là giữ nguyên các trọng số (weights) đã học sẵn của MobileNetV2, không cho nó cập nhật trong giai đoạn train đầu tiên. Việc này giúp tránh phá hỏng những gì nó đã học giỏi từ ImageNet, đồng thời train nhanh hơn (ít tham số cần cập nhật hơn).

In [ ]:
def build_model(num_classes, input_shape=(224, 224, 3)):
    # Tải MobileNetV2, bỏ phần "đầu phân loại" gốc (include_top=False)
    # vì phần đó được train để phân loại 1000 lớp của ImageNet, không phải 30 món ăn của mình
    base_model = MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights="imagenet"
    )

    # Đóng băng toàn bộ base model ở giai đoạn 1 (feature extraction)
    base_model.trainable = False

    inputs = tf.keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)
    return model, base_model

model, base_model = build_model(NUM_CLASSES)
model.summary()


## 6. Compile & Callbacks

**Compile** = cấu hình cách model sẽ học:
- `optimizer="adam"`: thuật toán cập nhật trọng số, Adam là lựa chọn mặc định tốt cho hầu hết bài toán
- `loss="categorical_crossentropy"`: hàm đo "model sai bao nhiêu" — dùng cho bài toán phân loại nhiều lớp (>2 lớp) với nhãn dạng one-hot (đúng với `class_mode="categorical"` ở trên)
- `metrics=["accuracy"]`: chỉ số để mình theo dõi, không dùng để tính gradient

**Callbacks** = các "trợ lý" tự động theo dõi quá trình train và can thiệp khi cần:

| Callback | Vai trò |
|---|---|
| `EarlyStopping` | Tự dừng train nếu `val_accuracy` không cải thiện sau N epoch liên tiếp → **chống overfitting**, đỡ tốn thời gian train vô ích (đây là thứ bản gốc của em bị thiếu, nên train đủ 10 epoch dù đã overfit từ epoch 4-5) |
| `ModelCheckpoint` | Tự lưu lại phiên bản model tốt nhất (val_accuracy cao nhất) trong quá trình train, phòng trường hợp epoch sau lại tệ đi |
| `ReduceLROnPlateau` | Tự giảm learning rate khi model "giậm chân tại chỗ", giúp model học tinh chỉnh hơn ở giai đoạn cuối |

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True,   # tự quay lại trọng số tốt nhất khi dừng
        verbose=1
    ),
    ModelCheckpoint(
        "best_model_stage1.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]


## 7. Huấn luyện — Giai đoạn 1 (Feature Extraction)

Ở giai đoạn này, MobileNetV2 vẫn đang **đóng băng**, chỉ có phần "đầu" mình tự thêm vào (Dense layers) được học. `EarlyStopping` sẽ tự dừng nếu không còn cải thiện, nên cứ để `epochs` hơi cao (ví dụ 20), model sẽ tự dừng sớm nếu cần — không cần lo train "dư" như bản gốc.

In [ ]:
EPOCHS_STAGE1 = 20

history_stage1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS_STAGE1,
    class_weight=class_weight_dict,
    callbacks=callbacks
)


## 8. Huấn luyện — Giai đoạn 2 (Fine-tuning)

### Fine-tuning là gì?
Sau khi phần "đầu" mới đã học tạm ổn, ta **rã đông (unfreeze)** một phần các lớp cuối của MobileNetV2 và train tiếp với **learning rate rất nhỏ**. Mục đích: để MobileNetV2 tinh chỉnh nhẹ những đặc trưng cấp cao của nó cho phù hợp hơn với món ăn Việt Nam (thay vì chỉ dừng ở đặc trưng chung chung của ImageNet).

⚠️ Vì sao learning rate phải **rất nhỏ** (`1e-5` thay vì `1e-3`)? Vì MobileNetV2 đã học rất tốt rồi, nếu học với tốc độ nhanh sẽ dễ "phá hỏng" kiến thức cũ (hiện tượng gọi là *catastrophic forgetting*). Ta chỉ muốn tinh chỉnh nhẹ nhàng.

Ta chỉ rã đông khoảng **1/3 số lớp cuối** của MobileNetV2 (thay vì toàn bộ), để giữ lại phần đặc trưng cơ bản (cạnh, màu sắc...) ở các lớp đầu — những đặc trưng này gần như không cần thay đổi dù là ảnh gì.

In [ ]:
base_model.trainable = True

# Chỉ rã đông (mở khoá) khoảng 1/3 số lớp cuối cùng của MobileNetV2
fine_tune_at = int(len(base_model.layers) * 0.66)

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Tổng số lớp trong MobileNetV2: {len(base_model.layers)}")
print(f"Số lớp được rã đông để fine-tune: {trainable_count}")

# Compile lại với learning rate NHỎ HƠN NHIỀU (bắt buộc phải compile lại sau khi đổi trainable)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
EPOCHS_STAGE2 = 15
# Train tiếp nối từ epoch cuối của giai đoạn 1
initial_epoch = history_stage1.epoch[-1] + 1

fine_tune_callbacks = [
    EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint("best_model_finetuned.keras", monitor="val_accuracy", save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-7, verbose=1)
]

history_stage2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=initial_epoch + EPOCHS_STAGE2,
    initial_epoch=initial_epoch,
    class_weight=class_weight_dict,
    callbacks=fine_tune_callbacks
)


## 9. Trực quan hoá quá trình huấn luyện

Vẽ đồ thị accuracy/loss là cách **nhanh nhất** để phát hiện overfitting:
- Nếu đường **train** đi lên tốt nhưng đường **val** đi ngang hoặc đi xuống → **overfitting** (giống bản gốc của em)
- Nếu cả 2 đường cùng đi lên và hội tụ gần nhau → model học tốt, tổng quát hoá ổn

In [ ]:
def plot_history(hist1, hist2=None):
    acc = hist1.history["accuracy"]
    val_acc = hist1.history["val_accuracy"]
    loss = hist1.history["loss"]
    val_loss = hist1.history["val_loss"]

    if hist2 is not None:
        acc += hist2.history["accuracy"]
        val_acc += hist2.history["val_accuracy"]
        loss += hist2.history["loss"]
        val_loss += hist2.history["val_loss"]

    epochs_range = range(len(acc))

    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label="Train Accuracy")
    plt.plot(epochs_range, val_acc, label="Validation Accuracy")
    if hist2 is not None:
        plt.axvline(x=len(hist1.history["accuracy"]) - 1, color="gray", linestyle="--", label="Bắt đầu Fine-tune")
    plt.legend(loc="lower right")
    plt.title("Accuracy qua từng epoch")
    plt.xlabel("Epoch")

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label="Train Loss")
    plt.plot(epochs_range, val_loss, label="Validation Loss")
    if hist2 is not None:
        plt.axvline(x=len(hist1.history["loss"]) - 1, color="gray", linestyle="--", label="Bắt đầu Fine-tune")
    plt.legend(loc="upper right")
    plt.title("Loss qua từng epoch")
    plt.xlabel("Epoch")

    plt.tight_layout()
    plt.show()

plot_history(history_stage1, history_stage2)


## 10. Đánh giá model trên tập Test

Tập **Test** là tập ảnh model **chưa từng nhìn thấy** trong suốt quá trình train/validate → đây là con số phản ánh trung thực nhất khả năng thực tế của model.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_data)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


In [ ]:
# Dự đoán toàn bộ tập test để tính chi tiết precision/recall/f1 cho từng món
test_data.reset()
y_pred_probs = model.predict(test_data, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_data.classes

idx_to_class = {v: k for k, v in test_data.class_indices.items()}
target_names = [idx_to_class[i] for i in range(NUM_CLASSES)]

print(classification_report(y_true, y_pred, target_names=target_names))


In [ ]:
# Confusion matrix - xem model hay nhầm lẫn giữa những món nào
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=False, cmap="YlOrRd", xticklabels=target_names, yticklabels=target_names)
plt.xlabel("Model dự đoán")
plt.ylabel("Nhãn thật")
plt.title("Confusion Matrix - Vietnamese Food AI")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


**Cách đọc confusion matrix:** mỗi ô (hàng i, cột j) thể hiện số ảnh thật sự thuộc món `i` nhưng bị model đoán thành món `j`. Đường chéo (từ trên-trái xuống dưới-phải) sáng màu = model đoán đúng nhiều. Nếu có ô ngoài đường chéo sáng bất thường → 2 món đó dễ bị nhầm lẫn với nhau (ví dụ có thể do hình dạng/màu sắc giống nhau, như Bánh bèo và Bánh khọt), em có thể thử tìm thêm ảnh phân biệt rõ hơn cho 2 món đó nếu muốn cải thiện.

## 11. Lưu model (để dùng cho Streamlit)

Ta cần lưu lại **2 thứ**:
1. **File model** (`.keras`) — chứa toàn bộ kiến trúc + trọng số đã học, để load lại dùng dự đoán sau này mà không cần train lại
2. **File `class_indices.json`** — vì model chỉ trả về số (ví dụ "lớp số 12"), mình cần file này để tra lại xem "lớp số 12" là món ăn gì

> 💡 Vì Google Colab sẽ **xoá hết file** sau khi hết phiên làm việc, nhớ tải file về máy hoặc lưu vào Google Drive để dùng lâu dài (ví dụ cho app Streamlit).

In [ ]:
MODEL_FILENAME = "vietnamese_food_mobilenetv2.keras"
CLASS_MAP_FILENAME = "class_indices.json"

model.save(MODEL_FILENAME)

# Lưu mapping: index (số) -> tên món ăn
idx_to_class = {v: k for k, v in train_data.class_indices.items()}
with open(CLASS_MAP_FILENAME, "w", encoding="utf-8") as f:
    json.dump(idx_to_class, f, ensure_ascii=False, indent=2)

print("Đã lưu:", MODEL_FILENAME)
print("Đã lưu:", CLASS_MAP_FILENAME)


In [ ]:
# (Tuỳ chọn) Lưu vào Google Drive để không bị mất khi Colab restart
# Bỏ comment 4 dòng dưới nếu em muốn lưu vào Drive:

# from google.colab import drive
# drive.mount('/content/drive')
# !cp {MODEL_FILENAME} /content/drive/MyDrive/
# !cp {CLASS_MAP_FILENAME} /content/drive/MyDrive/

# (Tuỳ chọn) Hoặc tải trực tiếp về máy tính cá nhân:
from google.colab import files
files.download(MODEL_FILENAME)
files.download(CLASS_MAP_FILENAME)


## 12. Demo dự đoán 1 ảnh

Đây chính là logic mà file `app.py` (Streamlit) sẽ dùng để dự đoán ảnh người dùng upload lên web.

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array

def predict_food(img_path, model, idx_to_class, top_k=3):
    img = load_img(img_path, target_size=IMG_SIZE)
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)   # PHẢI preprocess giống lúc train

    preds = model.predict(img_array, verbose=0)[0]
    top_indices = preds.argsort()[-top_k:][::-1]

    results = [(idx_to_class[i], float(preds[i])) for i in top_indices]

    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis("off")
    title = "\n".join([f"{name}: {prob*100:.1f}%" for name, prob in results])
    plt.title(title)
    plt.show()

    return results


# Thử với 1 ảnh ngẫu nhiên trong tập test
sample_class = random.choice(class_names)
sample_class_dir = os.path.join(test_dir, sample_class)
sample_img_name = random.choice(os.listdir(sample_class_dir))
sample_img_path = os.path.join(sample_class_dir, sample_img_name)

print("Nhãn thật:", sample_class)
predict_food(sample_img_path, model, idx_to_class, top_k=3)


## 13. Tổng kết & bước tiếp theo

### Tóm tắt những gì đã làm
- Chuyển từ CNN tự train từ đầu → **Transfer Learning MobileNetV2** (train nhanh hơn, chính xác hơn với dataset không quá lớn)
- Thêm **Data Augmentation** + **Dropout** + **EarlyStopping** → xử lý vấn đề overfitting nghiêm trọng của bản gốc
- Thêm **Fine-tuning** giai đoạn 2 để tăng độ chính xác thêm
- Thêm **class_weight** để xử lý mất cân bằng dữ liệu (nếu có)
- Đánh giá đầy đủ bằng `classification_report` + `confusion matrix`, không chỉ dừng ở accuracy
- Lưu model đúng cách (`.keras` + `class_indices.json`) để dùng lại cho web Streamlit

### Bước tiếp theo — Deploy lên Streamlit
File `app.py` đi kèm sẽ:
1. Load lại `vietnamese_food_mobilenetv2.keras` và `class_indices.json`
2. Cho người dùng upload ảnh món ăn
3. Hiển thị kết quả dự đoán + top-3 xác suất

Cách chạy thử ở local:
```bash
pip install streamlit tensorflow pillow
streamlit run app.py
```
Sau đó có thể deploy miễn phí lên [Streamlit Community Cloud](https://streamlit.io/cloud) bằng cách đẩy code lên GitHub rồi kết nối.

### Nếu muốn cải thiện thêm độ chính xác
- Thu thập thêm ảnh cho các món có ít ảnh nhất (xem lại bảng ở Bước 3)
- Thử các mạng transfer learning khác để so sánh: `EfficientNetB0`, `ResNet50V2`
- Áp dụng thêm kỹ thuật **Mixup** hoặc **CutMix** (augmentation nâng cao)
- Xem confusion matrix để tìm các cặp món hay bị nhầm, tìm hiểu tại sao (ảnh chụp góc giống nhau? màu sắc giống nhau?)
